# PRÉ-PROJETO DE PESQUISA OPERACIONAL — Python-MIP com HiGHS, CBC ou Gurobi

## Otimização Operacional do Agendamento de Bombas e Distribuição de Água Tratada via Programação Linear Inteira Mista (MILP)

- Escolha a resolução temporal na **Seção 2** alterando `MODO_TEMPORAL` (`"hora"` para 24 períodos ou `"minuto"` para 1.440 períodos).
- Escolha o solver na **Seção 3** alterando `SOLVER` (`"highs"`, `"highs_base"`, `"cbc"` ou `"gurobi"`).
- **Melhorias aplicadas:** formulação enxuta em `python-mip` (eliminação das variáveis redundantes `q` e das restrições explícitas de limites de `V`, reduzindo 35% das linhas), arcos pré-indexados, warm-start hierárquico opcional no modo minuto (`USAR_WARM_START` via `model.start`), acompanhamento temporal para **HiGHS e CBC** (tabela + gráficos de incumbente/limite inferior e gap), e proteção para execução independente do cenário-base.

## 1. Instalação de Dependências

Instala ou atualiza o `python-mip` (`mip==2.0.0`) e as bibliotecas dos solvers (`highspy`, `cbcbox` e `gurobipy`). Execute apenas na primeira configuração do ambiente.

In [ ]:
%pip install --upgrade "mip==2.0.0" "highspy>=1.8" cbcbox gurobipy

## 2. Configurações Gerais, Topologia e Dados do Sistema

Centraliza todas as chaves de controle (`MODO_TEMPORAL`, cenário-base, acompanhamento de progresso e warm-start), a topologia da rede (arcos bomba → reservatório e reservatório → setor), tarifas horossazonais, parâmetros físicos e curva de demanda diária.

In [ ]:
from collections import defaultdict
import mip
from mip.exceptions import InterfacingError
from mip import Model, xsum, BINARY, CONTINUOUS, MINIMIZE, OptimizationStatus
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ==============================================================================
# CONFIGURAÇÕES PRINCIPAIS DE EXECUÇÃO (altere aqui)
# ==============================================================================
# Resolução temporal do modelo: "hora" (24 intervalos) ou "minuto" (1.440 intervalos)
MODO_TEMPORAL = "minuto"

# Cenário-base (tarifa fixa): defina False para pular o cálculo e os gráficos do cenário-base.
GERAR_CENARIO_BASE = True
TEMPO_MAX_BASE_S = 300  # Limite próprio (s) para o cenário-base

# Acompanhamento da evolução do solver ao longo do tempo (HiGHS via log_file; CBC via captura do log)
ACOMPANHAR_PROGRESSO = True
INTERVALO_ACOMPANHAMENTO_MIN = 1  # Ex.: 5 ou 1 (no modo "hora" aceita valores menores, ex.: 0.1 = 6 s)

# Warm-start hierárquico (ativo automaticamente apenas no modo "minuto"):
# resolve primeiro o modelo agregado (ex.: 60 min = 24 passos) e injeta a solução via model.start.
USAR_WARM_START = True
PASSO_WARM_START_MIN = 60    # Deve dividir 60 (ex.: 60 ou 15)
TEMPO_MAX_WARM_START_S = 60  # 60 s permite ao modelo horário convergir ao ótimo (~R$ 10.724–10.733)

# Penalidade por partida (R$/acionamento). Mantenha 0.0 para máximo desempenho.
PENALIDADE_PARTIDA = 0.0

# Tempo mínimo contínuo que uma bomba deve permanecer ligada (em minutos).
# 0 mantém exatamente a formulação matemática original.
TEMPO_MIN_LIGADO_MIN = 0
# ==============================================================================

if MODO_TEMPORAL not in ("hora", "minuto"):
    raise ValueError(
        f"MODO_TEMPORAL inválido: {MODO_TEMPORAL!r}. Escolha 'hora' ou 'minuto'."
    )

# 1. Conjuntos de Índices derivados automaticamente do modo selecionado
T = 24
horas = range(T)
MINUTOS_POR_DIA = T * 60
DURACAO_PASSO_MIN = 60 if MODO_TEMPORAL == "hora" else 1
N_PASSOS = MINUTOS_POR_DIA // DURACAO_PASSO_MIN
FATOR_HORA = DURACAO_PASSO_MIN / 60.0
passos = range(N_PASSOS)
DESCRICAO_MODO = (
    f"passos de 1 hora ({N_PASSOS} períodos)"
    if MODO_TEMPORAL == "hora"
    else f"passos de 1 minuto ({N_PASSOS} períodos)"
)
bombas = range(1, 7)
reservatorios = range(1, 6)
setores = range(1, 13)

# 2. Topologia do Sistema (Arcos permitidos conforme o diagrama)
arcos_PR = [
    (1, 1),
    (2, 1), (2, 2), (2, 3),
    (3, 2),
    (4, 3), (4, 4),
    (5, 5),
    (6, 4), (6, 5)
]

arcos_RD = [
    (1, 1), (1, 2), (1, 3),
    (2, 2), (2, 4), (2, 5),
    (3, 4), (3, 6), (3, 7),
    (4, 4), (4, 8), (4, 9), (4, 10), (4, 12),
    (5, 7), (5, 10), (5, 11), (5, 12)
]

# Pré-indexação dos arcos para evitar buscas lineares durante a montagem do modelo
arcos_saida_bomba = defaultdict(list)
arcos_entrada_res = defaultdict(list)
for p, r in arcos_PR:
    arcos_saida_bomba[p].append((p, r))
    arcos_entrada_res[r].append((p, r))

arcos_saida_res = defaultdict(list)
arcos_entrada_setor = defaultdict(list)
for r, d in arcos_RD:
    arcos_saida_res[r].append((r, d))
    arcos_entrada_setor[d].append((r, d))

# 3. Parâmetros Econômicos (Tarifa Horossazonal - R$/kWh)
# Tarifa fora de ponta: R$ 0.4525 | Tarifa de ponta (18h00 às 20h59): R$ 1.4048
C = {t: 1.4048 if 18 <= t < 21 else 0.4525 for t in horas}

# 4. Parâmetros das bombas: (potência em kW, vazão mínima e máxima em m³/h)
param_bombas = {
    1: (240, 1170, 2340),
    2: (450, 2180, 4370),
    3: (240, 1170, 2340),
    4: (290, 1400, 2810),
    5: (190,  940, 1870),
    6: (240, 1170, 2340)
}
Q_ETA_max = 9400  # m³/h; limite de vazão agregada da ETA

# 5. Parâmetros dos Reservatórios: (volume mínimo, máximo e inicial), em m³
param_res = {
    1: (2100, 14000, 4200),
    2: (1400,  9400, 2800),
    3: (1400,  9400, 2800),
    4: (1400,  9400, 2800),
    5: ( 700,  4700, 1400)
}

# 6. Demandas-base dos setores (m³/h); total: 5195 m³/h
base_demanda = {
    1: 312,  2: 468,  3: 390,  4: 624,
    5: 390,  6: 312,  7: 546,  8: 468,
    9: 312, 10: 702, 11: 390, 12: 281
}

# Curva horária de demanda de verão; cada fator multiplica a demanda-base dos setores.
curva_verao = [
    0.8, 0.5, 0.3, 0.4, 0.6, 1.0,   # 00h às 05h: menor consumo na madrugada
    1.8, 2.0, 1.9, 1.7, 1.5, 1.5,   # 06h às 11h: pico da manhã
    1.4, 1.5, 1.6, 1.7, 1.8, 2.9,   # 12h às 17h: consumo alto e crescente
    3.4, 3.0, 2.6, 2.0, 1.5, 1.0    # 18h às 23h: pico máximo às 18h
]

D = {
    (d, t): round(base_demanda[d] * curva_verao[t], 2)
    for d in setores for t in horas
}

# Indicadores de demanda e armazenamento calculados para o horizonte diário.
demanda_horaria_total = [sum(D[d, h] for d in setores) for h in horas]
volume_demanda_diario = sum(demanda_horaria_total)
demanda_media_horaria = volume_demanda_diario / T
hora_pico = max(horas, key=lambda h: demanda_horaria_total[h])
demanda_pico_horaria = demanda_horaria_total[hora_pico]
armazenamento_util_total = sum(
    param_res[r][1] - param_res[r][0] for r in reservatorios
)

saldo_acumulado = [0.0]
for demanda in demanda_horaria_total:
    saldo_acumulado.append(
        saldo_acumulado[-1] + demanda_media_horaria - demanda
    )
armazenamento_util_estimado = max(saldo_acumulado) - min(saldo_acumulado)

def clamp(valor, limite_inf, limite_sup):
    """Garante que um valor numérico fique estritamente dentro dos bounds [limite_inf, limite_sup]."""
    return min(limite_sup, max(limite_inf, float(valor)))

def contar_partidas(x_vars, n_passos_modelo=None):
    """Conta o número total de transições 0 -> 1 nas bombas (funciona para modo hora ou minuto)."""
    n_p = n_passos_modelo if n_passos_modelo is not None else N_PASSOS
    total = 0
    for p in bombas:
        anterior = 0
        for t in range(n_p):
            val = x_vars[p, t].x
            atual = 1 if (val is not None and val > 0.5) else 0
            if atual == 1 and anterior == 0:
                total += 1
            anterior = atual
    return total

print(f"Modo selecionado: {MODO_TEMPORAL.upper()} — {DESCRICAO_MODO}")
print(f"Soma das demandas-base: {sum(base_demanda.values())} m³/h")
print(f"Demanda média horária: {demanda_media_horaria:.2f} m³/h")
print(f"Pico horário: {demanda_pico_horaria:.2f} m³/h às {hora_pico:02d}h")
print(f"Volume de demanda diário: {volume_demanda_diario:.2f} m³/dia")
print(f"Volume útil agregado dos reservatórios: {armazenamento_util_total:.0f} m³")
print(
    "Armazenamento útil estimado pelo perfil: "
    f"{armazenamento_util_estimado:.2f} m³ "
    "(estimativa agregada, com bombeamento constante na demanda média)"
)

## 3. Configuração do Solver e Construção do Modelo MILP

Define o backend do `python-mip` (`SOLVER`), parâmetros específicos de otimização (incluindo opções avançadas do HiGHS e validação do `GUROBI_HOME`), o construtor enxuto do modelo (`construir_modelo_eta`) e as rotinas de warm-start hierárquico (`gerar_warm_start`), construindo o modelo principal no modo escolhido.

In [ ]:
import os
import sys
import time
from pathlib import Path

# Altere somente esta variável: "highs", "highs_base", "cbc" ou "gurobi".
SOLVER = "cbc"

SOLVERS = {
    "highs": {
        "backend": mip.HIGHS,
        "time_limit": 300,  # Em segundos (3600 = 1 hora)
        "mip_rel_gap": 0.003,
        "threads": 0,
        "seed": 0,
        "highs_options": {
            "mip_detect_symmetry": True,
            "mip_heuristic_effort": 0.10,
            "mip_heuristic_run_shifting": True,
            "parallel": "choose",
            "random_seed": 0,
        },
    },
    "highs_base": {
        "backend": mip.HIGHS,
        "time_limit": 300,
        "mip_rel_gap": 0.003,
        "threads": 0,
        "seed": 0,
        "highs_options": {
            "parallel": "choose",
            "random_seed": 0,
        },
    },
    "cbc": {
        "backend": mip.CBC,
        "time_limit": 300,
        "mip_rel_gap": 0.003,
        "threads": 0,
        "seed": 0,
        "highs_options": {},
    },
    "gurobi": {
        "backend": mip.GRB,
        "time_limit": 300,
        "mip_rel_gap": 0.003,
        "threads": 0,
        "seed": 0,
        "highs_options": {},
    },
}

if SOLVER not in SOLVERS:
    raise ValueError(
        f"SOLVER inválido: {SOLVER!r}. Escolha entre {', '.join(SOLVERS)}."
    )

if SOLVER == "gurobi":
    gurobi_home = os.environ.get("GUROBI_HOME")
    if not gurobi_home:
        raise RuntimeError(
            "O backend Gurobi do python-mip usa a API C. Configure GUROBI_HOME "
            "para uma instalação nativa do Gurobi com licença compatível e reinicie o kernel. "
            "A licença restrita instalada pelo pacote pip gurobipy só pode ser usada pela interface gurobipy."
        )
    gurobi_home = Path(gurobi_home).expanduser()
    diretorio_bibliotecas = gurobi_home / ("bin" if os.name == "nt" else "lib")
    padrao_biblioteca = "gurobi*.dll" if os.name == "nt" else "libgurobi*"
    bibliotecas_nativas = {p.resolve() for p in diretorio_bibliotecas.glob(padrao_biblioteca)}
    if not bibliotecas_nativas:
        raise RuntimeError(
            f"Nenhuma biblioteca nativa do Gurobi ({padrao_biblioteca}) encontrada em "
            f"{diretorio_bibliotecas}. Confira o valor de GUROBI_HOME."
        )
    modulo_gurobi_mip = sys.modules.get("mip.gurobi")
    if modulo_gurobi_mip is not None:
        biblioteca_carregada = getattr(modulo_gurobi_mip, "lib_path", None)
        if biblioteca_carregada and Path(biblioteca_carregada).resolve() not in bibliotecas_nativas:
            raise RuntimeError(
                "O python-mip já carregou outra biblioteca Gurobi neste kernel. "
                "Reinicie o kernel depois de configurar GUROBI_HOME."
            )

def obter_gap_alvo_pct():
    """Retorna o gap-alvo configurado para o solver selecionado, em porcentagem."""
    gap = SOLVERS[SOLVER].get("mip_rel_gap")
    return float(gap) * 100.0 if gap is not None else None

def aplicar_opcoes_solver(m, verbose=False, time_limit_override=None, highs_log_file=None, highs_log_interval_s=None):
    """Configura limites, threads, verbosidade e opções avançadas do HiGHS no modelo python-mip."""
    cfg = SOLVERS[SOLVER]
    limite_s = time_limit_override if time_limit_override is not None else cfg["time_limit"]
    m.max_seconds = limite_s
    m.max_mip_gap = cfg["mip_rel_gap"]
    m.threads = cfg["threads"]
    m.seed = cfg["seed"]
    m.verbose = 1 if verbose else 0

    if SOLVER in ("highs", "highs_base"):
        solver_h = m.solver
        for opt, val in cfg["highs_options"].items():
            try:
                if isinstance(val, bool):
                    solver_h._set_bool_option_value(opt, val)
                elif isinstance(val, int):
                    solver_h._set_int_option_value(opt, val)
                elif isinstance(val, float):
                    solver_h._set_double_option_value(opt, val)
                elif isinstance(val, str):
                    solver_h._set_string_option_value(opt, val)
            except Exception:
                pass
        if highs_log_file:
            solver_h._set_string_option_value("log_file", highs_log_file)
        if highs_log_interval_s is not None:
            solver_h._set_double_option_value("mip_min_logging_interval", float(highs_log_interval_s))

def construir_modelo_eta(
    passo_min=1,
    penalidade_partida=0.0,
    tempo_min_ligado_min=0,
    tarifa_fixa=False,
    vazao_ponta_min_m3h=None,
    nome="Agendamento_Bombas_MILP",
    verbose=False,
    time_limit_override=None,
):
    """Constrói o MILP enxuto em python-mip para qualquer passo divisor de 60 min (ex.: 60 ou 1)."""
    if 60 % passo_min != 0:
        raise ValueError(f"passo_min ({passo_min}) deve ser um divisor de 60.")

    n_passos = MINUTOS_POR_DIA // passo_min
    fator_h = passo_min / 60.0

    try:
        m = Model(
            name=nome,
            sense=MINIMIZE,
            solver_name=SOLVERS[SOLVER]["backend"],
        )
    except InterfacingError as exc:
        if SOLVER == "gurobi":
            raise RuntimeError(
                "O python-mip não conseguiu iniciar o Gurobi pela API C. Verifique se GUROBI_HOME "
                "aponta para a instalação nativa correta e se a licença é compatível com a API C."
            ) from exc
        raise

    aplicar_opcoes_solver(m, verbose=verbose, time_limit_override=time_limit_override)

    # Variáveis com limites embutidos (elimina variáveis redundantes q e 14.405 restrições de V)
    x = {
        (p, t): m.add_var(var_type=BINARY, name=f"x_{p}_{t}")
        for p in bombas for t in range(n_passos)
    }
    f_pr = {
        (p, r, t): m.add_var(
            var_type=CONTINUOUS, lb=0.0, ub=param_bombas[p][2] / 60.0,
            name=f"f_pr_{p}_{r}_{t}",
        )
        for p, r in arcos_PR for t in range(n_passos)
    }
    f_rd = {
        (r, d, t): m.add_var(
            var_type=CONTINUOUS, lb=0.0, ub=D[d, (t * passo_min) // 60] / 60.0,
            name=f"f_rd_{r}_{d}_{t}",
        )
        for r, d in arcos_RD for t in range(n_passos)
    }
    V = {}
    for r in reservatorios:
        v_min, v_max, v_ini = param_res[r]
        for t in range(n_passos):
            lb = max(v_min, v_ini) if t == n_passos - 1 else v_min
            V[r, t] = m.add_var(var_type=CONTINUOUS, lb=lb, ub=v_max, name=f"V_{r}_{t}")

    # Detecção opcional de partidas (somente se penalidade_partida > 0 ou tempo_min_ligado_min > passo_min)
    passos_min_ligado = int(np.ceil(tempo_min_ligado_min / passo_min)) if tempo_min_ligado_min > 1 else 0
    usar_partidas = (penalidade_partida > 0) or (passos_min_ligado > 1)
    y = None
    if usar_partidas:
        y = {
            (p, t): m.add_var(var_type=CONTINUOUS, lb=0.0, ub=1.0, name=f"y_{p}_{t}")
            for p in bombas for t in range(n_passos)
        }
        for p in bombas:
            m += y[p, 0] >= x[p, 0], f"Partida_{p}_0"
            for t in range(1, n_passos):
                m += y[p, t] >= x[p, t] - x[p, t - 1], f"Partida_{p}_{t}"
    if passos_min_ligado > 1:
        for p in bombas:
            for t in range(n_passos):
                inicio = max(0, t - passos_min_ligado + 1)
                m += xsum(y[p, tau] for tau in range(inicio, t + 1)) <= x[p, t], f"MinLig_{p}_{t}"

    # Objetivo
    if tarifa_fixa:
        obj_expr = xsum(
            param_bombas[p][0] * fator_h * x[p, t]
            for p in bombas for t in range(n_passos)
        )
    else:
        obj_expr = xsum(
            C[(t * passo_min) // 60] * param_bombas[p][0] * fator_h * x[p, t]
            for p in bombas for t in range(n_passos)
        )
    if penalidade_partida > 0 and y is not None:
        m.objective = obj_expr + penalidade_partida * xsum(
            y[p, t] for p in bombas for t in range(n_passos)
        )
    else:
        m.objective = obj_expr

    # Limites operacionais das bombas (q expresso diretamente como soma de f_pr)
    for p in bombas:
        q_min_m = param_bombas[p][1] / 60.0
        q_max_m = param_bombas[p][2] / 60.0
        arcos_p = arcos_saida_bomba[p]
        for t in range(n_passos):
            q_pt = xsum(f_pr[p, r, t] for _, r in arcos_p)
            m += q_pt >= q_min_m * x[p, t], f"Bomba_Min_{p}_{t}"
            m += q_pt <= q_max_m * x[p, t], f"Bomba_Max_{p}_{t}"

    # Capacidade máxima da ETA e restrição opcional de ponta (cenário-base)
    q_eta_m = Q_ETA_max / 60.0
    for t in range(n_passos):
        vazao_total_t = xsum(f_pr[p, r, t] for p, r in arcos_PR)
        m += vazao_total_t <= q_eta_m, f"Capacidade_ETA_{t}"
        hora = (t * passo_min) // 60
        if vazao_ponta_min_m3h is not None and hora in vazao_ponta_min_m3h:
            m += vazao_total_t >= vazao_ponta_min_m3h[hora] / 60.0 - 1e-6, f"Vazao_Ponta_{t}"

    # Atendimento integral da demanda
    for d in setores:
        arcos_d = arcos_entrada_setor[d]
        for t in range(n_passos):
            m += xsum(f_rd[r, d, t] for r, _ in arcos_d) == D[d, (t * passo_min) // 60] / 60.0, f"Demanda_{d}_{t}"

    # Balanço hídrico dos reservatórios
    for r in reservatorios:
        v_ini = param_res[r][2]
        ent_r = arcos_entrada_res[r]
        sai_r = arcos_saida_res[r]
        for t in range(n_passos):
            v_ant = v_ini if t == 0 else V[r, t - 1]
            m += (
                V[r, t] == v_ant
                + passo_min * xsum(f_pr[p, r, t] for p, _ in ent_r)
                - passo_min * xsum(f_rd[r, d, t] for _, d in sai_r)
            ), f"Balanco_{r}_{t}"

    variaveis = {
        "x": x, "f_pr": f_pr, "f_rd": f_rd, "V": V, "y": y,
        "passo_min": passo_min, "n_passos": n_passos,
    }
    return m, variaveis

def aplicar_warm_start(modelo_destino, vars_destino, vars_agregado, passo_destino_min, passo_agregado_min):
    """Expande a solução viável do modelo agregado e define modelo_destino.start."""
    fator_exp = passo_agregado_min // passo_destino_min
    n_dest = MINUTOS_POR_DIA // passo_destino_min
    lista_start = []

    x_dest, f_pr_dest, f_rd_dest, V_dest, y_dest = (
        vars_destino["x"], vars_destino["f_pr"], vars_destino["f_rd"], vars_destino["V"], vars_destino["y"]
    )
    x_ag, f_pr_ag, f_rd_ag = vars_agregado["x"], vars_agregado["f_pr"], vars_agregado["f_rd"]

    for p in bombas:
        anterior = 0
        for t in range(n_dest):
            x_val = int(round(x_ag[p, t // fator_exp].x))
            lista_start.append((x_dest[p, t], float(x_val)))
            if y_dest is not None:
                lista_start.append((y_dest[p, t], 1.0 if (x_val == 1 and anterior == 0) else 0.0))
            anterior = x_val

    f_pr_vals = {}
    for p, r in arcos_PR:
        ub = param_bombas[p][2] / 60.0
        for t in range(n_dest):
            val = clamp(f_pr_ag[p, r, t // fator_exp].x, 0.0, ub)
            f_pr_vals[p, r, t] = val
            lista_start.append((f_pr_dest[p, r, t], val))

    f_rd_vals = {}
    for r, d in arcos_RD:
        for t in range(n_dest):
            ub = D[d, (t * passo_destino_min) // 60] / 60.0
            val = clamp(f_rd_ag[r, d, t // fator_exp].x, 0.0, ub)
            f_rd_vals[r, d, t] = val
            lista_start.append((f_rd_dest[r, d, t], val))

    for r in reservatorios:
        v_min, v_max, v_ini = param_res[r]
        v_atual = float(v_ini)
        for t in range(n_dest):
            entrada = sum(f_pr_vals[p, r, t] for p, _ in arcos_entrada_res[r])
            saida = sum(f_rd_vals[r, d, t] for _, d in arcos_saida_res[r])
            v_atual = v_atual + passo_destino_min * (entrada - saida)
            lb = max(v_min, v_ini) if t == n_dest - 1 else v_min
            lista_start.append((V_dest[r, t], clamp(v_atual, lb, v_max)))

    modelo_destino.start = lista_start

def gerar_warm_start(modelo_destino, vars_destino, tarifa_fixa=False, vazao_ponta_min_m3h=None):
    """Resolve o modelo agregado (PASSO_WARM_START_MIN) e injeta a solução via model.start."""
    if not USAR_WARM_START:
        return False
    if DURACAO_PASSO_MIN >= PASSO_WARM_START_MIN:
        print(
            f"[WARM-START] Modo {MODO_TEMPORAL} ativo (passo {DURACAO_PASSO_MIN} min); "
            "warm-start agregado não é necessário."
        )
        return False
    if PASSO_WARM_START_MIN % DURACAO_PASSO_MIN != 0:
        raise ValueError(
            f"PASSO_WARM_START_MIN ({PASSO_WARM_START_MIN}) deve ser múltiplo de DURACAO_PASSO_MIN ({DURACAO_PASSO_MIN})."
        )

    t_ws = time.perf_counter()
    modelo_ws, vars_ws = construir_modelo_eta(
        passo_min=PASSO_WARM_START_MIN,
        penalidade_partida=PENALIDADE_PARTIDA,
        tempo_min_ligado_min=TEMPO_MIN_LIGADO_MIN,
        tarifa_fixa=tarifa_fixa,
        vazao_ponta_min_m3h=vazao_ponta_min_m3h,
        nome=f"WarmStart_{PASSO_WARM_START_MIN}min",
        verbose=False,
        time_limit_override=TEMPO_MAX_WARM_START_S,
    )
    status_ws = modelo_ws.optimize(max_seconds=TEMPO_MAX_WARM_START_S)
    if status_ws not in (OptimizationStatus.OPTIMAL, OptimizationStatus.FEASIBLE) or modelo_ws.objective_value is None:
        print(f"[WARM-START] Nenhuma solução viável encontrada em {TEMPO_MAX_WARM_START_S} s (status: {status_ws}).")
        return False

    aplicar_warm_start(modelo_destino, vars_destino, vars_ws, DURACAO_PASSO_MIN, PASSO_WARM_START_MIN)
    duracao_ws = time.perf_counter() - t_ws
    unidade = "kWh" if tarifa_fixa else "R$"
    print(
        f"[WARM-START {PASSO_WARM_START_MIN} min -> {DURACAO_PASSO_MIN} min] "
        f"Solução inicial viável gerada em {duracao_ws:.2f} s "
        f"(status: {status_ws} | objetivo: {modelo_ws.objective_value:.2f} {unidade})."
    )
    return True

# Construção do modelo principal no modo selecionado (hora ou minuto)
t_construcao_ini = time.perf_counter()
model, vars_model = construir_modelo_eta(
    passo_min=DURACAO_PASSO_MIN,
    penalidade_partida=PENALIDADE_PARTIDA,
    tempo_min_ligado_min=TEMPO_MIN_LIGADO_MIN,
    tarifa_fixa=False,
    nome=f"Agendamento_Bombas_{SOLVER.upper()}_{MODO_TEMPORAL.upper()}",
    verbose=False,
)

print(
    f"Solver selecionado: {SOLVER.upper()} (python-mip) | "
    f"Modelo ({DESCRICAO_MODO}) construído em {time.perf_counter() - t_construcao_ini:.2f} s."
)

## 4. Cenário-Base com Tarifa Fixa (Opcional)

Calcula a operação de referência sem sinal de preço horário (mantendo vazão mínima na ponta igual à demanda agregada, limitada à capacidade da ETA) e recalcula o custo com as tarifas reais. Esta célula pode ser pulada ou desativada (`GERAR_CENARIO_BASE = False`) sem afetar o restante do notebook.

In [ ]:
# O cenário-base é opcional; a otimização tarifária não depende dele.
modelo_base = None
vars_base = None
status_base = None
consumo_base_kwh = None
custo_base_real = None
partidas_base = None
vazao_base_ponta = None
cenario_base_disponivel = False

if GERAR_CENARIO_BASE:
    horas_ponta = [h for h in horas if C[h] == max(C.values())]
    vazao_base_ponta = {
        h: min(sum(D[d, h] for d in setores), Q_ETA_max)
        for h in horas_ponta
    }

    modelo_base, vars_base = construir_modelo_eta(
        passo_min=DURACAO_PASSO_MIN,
        penalidade_partida=PENALIDADE_PARTIDA,
        tempo_min_ligado_min=TEMPO_MIN_LIGADO_MIN,
        tarifa_fixa=True,
        vazao_ponta_min_m3h=vazao_base_ponta,
        nome=f"Cenario_Base_{SOLVER.upper()}_{MODO_TEMPORAL.upper()}",
        verbose=False,
        time_limit_override=TEMPO_MAX_BASE_S,
    )

    gerar_warm_start(
        modelo_base, vars_base, tarifa_fixa=True, vazao_ponta_min_m3h=vazao_base_ponta
    )

    status_base = modelo_base.optimize(max_seconds=TEMPO_MAX_BASE_S)
    cenario_base_disponivel = (
        status_base in (OptimizationStatus.OPTIMAL, OptimizationStatus.FEASIBLE)
        and modelo_base.objective_value is not None
    )

    if not cenario_base_disponivel:
        print(
            "O cenário-base não encontrou solução viável. "
            f"Status: {status_base}. Comparação e gráficos do cenário-base serão ignorados."
        )
    else:
        x_base = vars_base["x"]
        consumo_base_kwh = sum(
            param_bombas[p][0] * FATOR_HORA * x_base[p, t].x
            for p in bombas for t in passos
        )
        custo_base_real = sum(
            C[(t * DURACAO_PASSO_MIN) // 60] * param_bombas[p][0] * FATOR_HORA * x_base[p, t].x
            for p in bombas for t in passos
        )
        partidas_base = contar_partidas(x_base, N_PASSOS)
        print(f"[CENÁRIO-BASE: TARIFA FIXA — {DESCRICAO_MODO}]")
        print("Status:", status_base)
        print("Vazão-alvo na ponta (m³/h):", vazao_base_ponta)
        print(f"Consumo de energia: {consumo_base_kwh:.2f} kWh")
        print(f"Custo dessa programação com as tarifas reais: R$ {custo_base_real:.2f}")
        print(f"Total de partidas das bombas: {partidas_base}")
else:
    print("Cálculo do cenário-base desativado; a otimização tarifária continuará normalmente.")

## 5. Otimização Tarifária, Acompanhamento e Comparação de Custos

Resolve o MILP principal em `python-mip` com warm-start opcional, coleta a evolução temporal do solver (**HiGHS** via `log_file` nativo e **CBC** via captura em tempo real do log de Branch-and-Bound), exibe a tabela de progresso e compara os resultados com o cenário-base quando disponível.

In [ ]:
# Otimização tarifária, acompanhamento temporal (HiGHS e CBC) e comparação de custos
import re
import tempfile
import threading

historico_progresso = []
tabela_progresso = None

# Warm-start opcional a partir do modelo agregado (ativo automaticamente apenas no modo "minuto")
gerar_warm_start(model, vars_model, tarifa_fixa=False)

def numero_finito(valor):
    if valor is None:
        return None
    try:
        numero = float(valor)
    except (TypeError, ValueError):
        return None
    return numero if np.isfinite(numero) else None

def base_esta_disponivel():
    """Verifica com segurança se os dados do cenário-base existem (mesmo se a Seção 4 não foi executada)."""
    return (
        bool(globals().get("cenario_base_disponivel", False))
        and globals().get("vars_base") is not None
        and globals().get("consumo_base_kwh") is not None
        and globals().get("custo_base_real") is not None
    )

def capturar_stdout_fd(fn, mostrar_ao_vivo=True):
    """Captura saída em nível de file descriptor (C/C++), mantendo exibição ao vivo."""
    sys.stdout.flush()
    try:
        orig_fd = os.dup(1)
    except OSError:
        return fn(), ""

    r_fd, w_fd = os.pipe()
    os.dup2(w_fd, 1)
    os.close(w_fd)
    chunks = []

    def leitor():
        while True:
            data = os.read(r_fd, 4096)
            if not data:
                break
            chunks.append(data)
            if mostrar_ao_vivo:
                try:
                    sys.stdout.write(data.decode("utf-8", errors="ignore"))
                    sys.stdout.flush()
                except Exception:
                    try:
                        os.write(orig_fd, data)
                    except OSError:
                        pass
        os.close(r_fd)

    th = threading.Thread(target=leitor, daemon=True)
    th.start()
    try:
        res = fn()
    finally:
        sys.stdout.flush()
        os.dup2(orig_fd, 1)
        os.close(orig_fd)
        th.join(timeout=5)
    return res, b"".join(chunks).decode("utf-8", errors="ignore")

def extrair_progresso_log_highs(texto_log, intervalo_s):
    """Extrai série temporal, nós, tempo interno e última melhoria a partir do log do HiGHS."""
    if not texto_log:
        return [], {"tempo_s": None, "custo": None}, None, None

    # Ex.: " C       0       0         0   0.00%   10676.038148    12437.6899        14.16%    10730   3438      0     46801    44.8s"
    regex_linha = re.compile(
        r"^\s*([A-Za-z])?\s+(\d+)\s+(\d+)\s+(\d+)\s+([0-9.]+%)\s+"
        r"([+-]?\d+(?:\.\d+)?(?:[eE][+-]?\d+)?|-?inf)\s+"
        r"([+-]?\d+(?:\.\d+)?(?:[eE][+-]?\d+)?|-?inf)\s+"
        r"([0-9.]+%|Large|inf)\s+"
        r"(\d+)\s+(\d+)\s+(\d+)\s+(\d+)\s+([0-9.]+)s\s*$"
    )
    pontos = []
    melhor_info = {"tempo_s": None, "custo": None}
    melhor_custo_visto = None

    for linha in texto_log.splitlines():
        m = regex_linha.match(linha)
        if not m:
            continue
        src, proc_str, _inq, _leaves, _expl, bnd_str, sol_str, gap_str, _cuts, _inlp, _confl, _iters, tempo_str = m.groups()
        t_s = numero_finito(tempo_str)
        custo = numero_finito(sol_str)
        limite = numero_finito(bnd_str)
        gap_pct = numero_finito(gap_str.rstrip("%"))
        if gap_pct is None and custo is not None and limite is not None and abs(custo) > 1e-12:
            gap_pct = abs(custo - limite) / abs(custo) * 100.0
        if t_s is not None:
            pontos.append((t_s, custo, limite, gap_pct, int(proc_str)))
            if custo is not None and (melhor_custo_visto is None or custo < melhor_custo_visto - 1e-7):
                melhor_custo_visto = custo
                melhor_info = {"tempo_s": t_s, "custo": custo}

    # Checa se MIP start foi aceito no início
    m_start = re.search(r"MIP start solution is feasible:\s*objective value is\s*([+-]?\d+(?:\.\d+)?)", texto_log)
    if m_start:
        custo_ws = numero_finito(m_start.group(1))
        if custo_ws is not None and (melhor_custo_visto is None or custo_ws <= melhor_custo_visto + 1e-6):
            if melhor_info["custo"] is None or abs(melhor_info["custo"] - custo_ws) < 1e-5:
                melhor_info = {"tempo_s": 0.0, "custo": custo_ws}

    historico = []
    proxima_amostra_s = 0.0
    for t_s, custo, limite, gap_pct, nos_pt in pontos:
        if t_s >= proxima_amostra_s:
            historico.append({
                "Tempo (min)": t_s / 60.0,
                "Melhor custo viável (R$)": custo,
                "Limite inferior (R$)": limite,
                "Gap (%)": gap_pct,
                "Nós": nos_pt,
            })
            proxima_amostra_s = (int(t_s // intervalo_s) + 1) * intervalo_s

    m_timing = re.search(r"^\s*Timing\s+([0-9.]+)\s*$", texto_log, re.MULTILINE)
    tempo_solver = numero_finito(m_timing.group(1)) if m_timing else None
    m_nodes = re.search(r"^\s*Nodes\s+(\d+)\s*$", texto_log, re.MULTILINE)
    nos_final = int(m_nodes.group(1)) if m_nodes else (pontos[-1][4] if pontos else None)

    return historico, melhor_info, tempo_solver, nos_final

def extrair_progresso_log_cbc(texto_log, intervalo_s):
    """Extrai série temporal, nós, tempo interno, limite inferior e última melhoria do log do CBC."""
    if not texto_log:
        return [], {"tempo_s": None, "custo": None}, None, None, None

    limite_inf = None
    for padrao in (
        r"Bound:\s*([+-]?\d+(?:\.\d+)?(?:[eE][+-]?\d+)?)",
        r"\(best possible\s+([+-]?\d+(?:\.\d+)?(?:[eE][+-]?\d+)?)\)",
        r"Lower bound:\s*([+-]?\d+(?:\.\d+)?(?:[eE][+-]?\d+)?)",
    ):
        matches = re.findall(padrao, texto_log)
        if matches:
            val = numero_finito(matches[-1])
            if val is not None:
                limite_inf = val
                break

    regex_devel = re.compile(
        r"^\s*(★|\*)?\s*(\d+)\s+(\d+)\s+(\d+)\s+"
        r"([+-]?\d+(?:\.\d+)?(?:[eE][+-]?\d+)?|--|inf|-)\s+"
        r"(?:([A-Za-z0-9_/-]+)\s+)?"
        r"([+-]?\d+(?:\.\d+)?(?:[eE][+-]?\d+)?)\s+"
        r"([+-]?\d+(?:\.\d+)?%|--|inf)\s+"
        r"([+-]?\d+(?:\.\d+)?)\s*$"
    )
    regex_classico = re.compile(
        r"After\s+(\d+)\s+nodes,\s+\d+\s+on\s+tree,\s+"
        r"([+-]?\d+(?:\.\d+)?(?:[eE][+-]?\d+)?)\s+best\s+solution,\s+"
        r"best\s+possible\s+([+-]?\d+(?:\.\d+)?(?:[eE][+-]?\d+)?)\s+"
        r"\(([+-]?\d+(?:\.\d+)?)\s+seconds\)"
    )

    pontos = []
    melhor_info = {"tempo_s": None, "custo": None}
    melhor_custo_visto = None
    em_bb = False

    for linha in texto_log.splitlines():
        linha_low = linha.lower()
        if "branch and bound" in linha_low:
            em_bb = True
            continue
        if em_bb and ("postprocessing" in linha_low or linha.lstrip().startswith("✔")):
            em_bb = False

        m_dev = regex_devel.match(linha) if em_bb else None
        if m_dev:
            estrela, nos_str, _ontree, _depth, sol_str, _metodo, bnd_str, gap_str, tempo_str = m_dev.groups()
            t_s = numero_finito(tempo_str)
            custo = numero_finito(sol_str)
            limite = numero_finito(bnd_str)
            gap_pct = numero_finito(gap_str.rstrip("%"))
            if gap_pct is None and custo is not None and limite is not None and abs(custo) > 1e-12:
                gap_pct = abs(custo - limite) / abs(custo) * 100.0
            if t_s is not None:
                pontos.append((t_s, custo, limite, gap_pct, int(nos_str)))
                if custo is not None and (
                    estrela is not None or melhor_custo_visto is None or custo < melhor_custo_visto - 1e-7
                ):
                    melhor_custo_visto = custo
                    melhor_info = {"tempo_s": t_s, "custo": custo}
            continue

        m_cls = regex_classico.search(linha)
        if m_cls:
            nos_str, sol_str, bnd_str, tempo_str = m_cls.groups()
            t_s = numero_finito(tempo_str)
            custo = numero_finito(sol_str)
            limite = numero_finito(bnd_str)
            gap_pct = (
                abs(custo - limite) / abs(custo) * 100.0
                if (custo is not None and limite is not None and abs(custo) > 1e-12)
                else None
            )
            if t_s is not None:
                pontos.append((t_s, custo, limite, gap_pct, int(nos_str)))
                if custo is not None and (melhor_custo_visto is None or custo < melhor_custo_visto - 1e-7):
                    melhor_custo_visto = custo
                    melhor_info = {"tempo_s": t_s, "custo": custo}

    historico = []
    proxima_amostra_s = 0.0
    for t_s, custo, limite, gap_pct, nos_pt in pontos:
        if t_s >= proxima_amostra_s:
            historico.append({
                "Tempo (min)": t_s / 60.0,
                "Melhor custo viável (R$)": custo,
                "Limite inferior (R$)": limite,
                "Gap (%)": gap_pct,
                "Nós": nos_pt,
            })
            proxima_amostra_s = (int(t_s // intervalo_s) + 1) * intervalo_s

    m_wall = re.search(r"Total time \(Wallclock seconds\):\s*([0-9.]+)", texto_log)
    tempo_solver = numero_finito(m_wall.group(1)) if m_wall else None
    m_nodes = re.search(r"Enumerated nodes:\s*(\d+)", texto_log)
    nos_final = int(m_nodes.group(1)) if m_nodes else (pontos[-1][4] if pontos else None)

    return historico, melhor_info, tempo_solver, nos_final, limite_inf

if ACOMPANHAR_PROGRESSO and INTERVALO_ACOMPANHAMENTO_MIN <= 0:
    raise ValueError("INTERVALO_ACOMPANHAMENTO_MIN deve ser maior que zero.")

intervalo_amostragem_s = float(INTERVALO_ACOMPANHAMENTO_MIN) * 60.0
tempo_limite_principal = SOLVERS[SOLVER]["time_limit"]
caminho_log_highs = None
texto_log_solver = ""

if SOLVER in ("highs", "highs_base"):
    fd_h, caminho_log_highs = tempfile.mkstemp(prefix="highs_mip_", suffix=".log")
    os.close(fd_h)
    aplicar_opcoes_solver(
        model,
        verbose=True,
        time_limit_override=tempo_limite_principal,
        highs_log_file=caminho_log_highs,
        highs_log_interval_s=min(5.0, max(1.0, intervalo_amostragem_s)),
    )
else:
    aplicar_opcoes_solver(model, verbose=True, time_limit_override=tempo_limite_principal)

tempo_inicio = time.perf_counter()
try:
    if SOLVER == "cbc" and ACOMPANHAR_PROGRESSO:
        status_terminal, texto_log_solver = capturar_stdout_fd(
            lambda: model.optimize(max_seconds=tempo_limite_principal),
            mostrar_ao_vivo=True,
        )
    else:
        status_terminal = model.optimize(max_seconds=tempo_limite_principal)
        if caminho_log_highs and os.path.exists(caminho_log_highs):
            with open(caminho_log_highs, "r", encoding="utf-8", errors="ignore") as arq:
                texto_log_solver = arq.read()
finally:
    if caminho_log_highs and os.path.exists(caminho_log_highs):
        try:
            os.remove(caminho_log_highs)
        except OSError:
            pass

tempo_execucao = time.perf_counter() - tempo_inicio

solucao_disponivel = (
    status_terminal in (OptimizationStatus.OPTIMAL, OptimizationStatus.FEASIBLE)
    and model.objective_value is not None
)

incumbente = numero_finito(model.objective_value) if solucao_disponivel else None
limite_inferior = numero_finito(model.objective_bound) if solucao_disponivel else None
melhor_incumbente_info = {"tempo_s": None, "custo": None}
tempo_solver = None
nos = None

if SOLVER in ("highs", "highs_base"):
    hist_h, melhor_h, tempo_solver, nos = extrair_progresso_log_highs(
        texto_log_solver, intervalo_amostragem_s
    )
    if ACOMPANHAR_PROGRESSO:
        historico_progresso.extend(hist_h)
    if melhor_h["tempo_s"] is not None:
        melhor_incumbente_info = melhor_h
elif SOLVER == "cbc":
    hist_c, melhor_c, tempo_solver, nos, limite_log_c = extrair_progresso_log_cbc(
        texto_log_solver, intervalo_amostragem_s
    )
    if ACOMPANHAR_PROGRESSO:
        historico_progresso.extend(hist_c)
    if melhor_c["tempo_s"] is not None:
        melhor_incumbente_info = melhor_c
    if limite_inferior is None and limite_log_c is not None:
        limite_inferior = limite_log_c

tempo = tempo_solver if tempo_solver is not None else tempo_execucao

if limite_inferior is not None and incumbente is not None and abs(incumbente) > 1e-12:
    gap_final = abs(incumbente - limite_inferior) / abs(incumbente)
else:
    gap_final = numero_finito(model.gap) if solucao_disponivel else None

print(f"\n[RESULTADOS DO {SOLVER.upper()} VIA PYTHON-MIP — {DESCRICAO_MODO}]")
print("Status:", status_terminal)
print(f"Função objetivo incumbente: R$ {incumbente:.4f}" if incumbente is not None else "Incumbente: não disponível")
print(f"Melhor limite inferior: R$ {limite_inferior:.4f}" if limite_inferior is not None else "Limite inferior: não disponível")
print(f"Gap relativo final: {gap_final:.2%}" if gap_final is not None else "Gap relativo final: não disponível")
print(f"Tempo (solver): {tempo:.2f} s ({tempo / 60.0:.2f} min)")
if tempo_solver is not None and abs(tempo_execucao - tempo_solver) > 1.0:
    print(f"Tempo total da célula (incl. interface python-mip): {tempo_execucao:.2f} s")
if melhor_incumbente_info["tempo_s"] is not None:
    t_melhor = melhor_incumbente_info["tempo_s"]
    print(
        f"Última melhoria da incumbente: R$ {melhor_incumbente_info['custo']:.4f} "
        f"aos {t_melhor:.2f} s ({t_melhor / 60.0:.2f} min)"
    )
print(f"Nós explorados: {nos}" if nos is not None else "Nós explorados: não disponível")

if ACOMPANHAR_PROGRESSO:
    tempo_final_min = tempo / 60.0
    if historico_progresso:
        tempo_final_min = max(tempo_final_min, historico_progresso[-1]["Tempo (min)"])

    linha_final = {
        "Tempo (min)": tempo_final_min,
        "Melhor custo viável (R$)": float(incumbente) if incumbente is not None else None,
        "Limite inferior (R$)": float(limite_inferior) if limite_inferior is not None else None,
        "Gap (%)": gap_final * 100.0 if gap_final is not None else None,
        "Nós": int(nos) if nos is not None else None,
    }
    if not historico_progresso or (
        linha_final["Tempo (min)"] - historico_progresso[-1]["Tempo (min)"] > 1e-2
    ):
        historico_progresso.append(linha_final)
    else:
        historico_progresso[-1] = linha_final

    tabela_progresso = pd.DataFrame(historico_progresso)
    display(tabela_progresso.round({
        "Tempo (min)": 2,
        "Melhor custo viável (R$)": 2,
        "Limite inferior (R$)": 2,
        "Gap (%)": 3,
    }))

# Resumo de custos e comparação segura com o cenário-base (não quebra se a Seção 4 não foi executada)
if solucao_disponivel:
    print(f"\n[SUCESSO] Solução encontrada pelo {SOLVER.upper()} via python-mip!")
    x_otim = vars_model["x"]
    consumo_otimizado_kwh = sum(
        param_bombas[p][0] * FATOR_HORA * x_otim[p, t].x
        for p in bombas for t in passos
    )
    custo_otimizado_real = sum(
        C[(t * DURACAO_PASSO_MIN) // 60] * param_bombas[p][0] * FATOR_HORA * x_otim[p, t].x
        for p in bombas for t in passos
    )
    partidas_otimizado = contar_partidas(x_otim, N_PASSOS)

    base_ok = base_esta_disponivel()
    linhas_comparacao = []
    if base_ok:
        linhas_comparacao.append({
            "Cenário": f"Base (tarifa fixa · {MODO_TEMPORAL})",
            "Energia (kWh)": round(consumo_base_kwh, 2),
            "Custo com tarifa real (R$)": round(custo_base_real, 2),
            "Partidas das bombas": globals().get("partidas_base"),
        })
    linhas_comparacao.append({
        "Cenário": f"Otimização (tarifa horária · {MODO_TEMPORAL})",
        "Energia (kWh)": round(consumo_otimizado_kwh, 2),
        "Custo com tarifa real (R$)": round(custo_otimizado_real, 2),
        "Partidas das bombas": partidas_otimizado,
    })
    comparacao_custos = pd.DataFrame(linhas_comparacao)
    display(comparacao_custos)

    if base_ok:
        economia = custo_base_real - custo_otimizado_real
        percentual_economia = 100.0 * economia / custo_base_real if custo_base_real else 0.0
        print(f"Economia financeira: R$ {economia:.2f} ({percentual_economia:.2f}%)")
    else:
        print(
            "Cenário-base indisponível (desativado, não executado ou sem solução viável); "
            "a comparação de custos foi omitida."
        )
else:
    print(f"\nModelo inviável ou sem solução viável encontrada pelo {SOLVER.upper()}.")

## 6. Gráficos e Análise Visual

Gera os perfis de bombeamento, a dinâmica dos reservatórios (em m³ e %) para a otimização tarifária (e para o cenário-base, se disponível) e, com `ACOMPANHAR_PROGRESSO = True`, os gráficos separados da evolução de **Incumbente vs. Limite Inferior** e **Gap (%)** ao longo do tempo.

In [ ]:
# Gráficos adaptados automaticamente ao MODO_TEMPORAL ("hora" ou "minuto")
# + gráficos separados de progressão (incumbente/limite inferior e gap) quando ACOMPANHAR_PROGRESSO=True.
if globals().get("solucao_disponivel", False):
    tempo_horas_com_fim = [t * DURACAO_PASSO_MIN / 60.0 for t in range(N_PASSOS + 1)]
    preco_energia = [C[(t * DURACAO_PASSO_MIN) // 60] for t in passos] + [C[23]]
    cores_res = ["#2E7D32", "#689F38", "#AFB42B", "#FBC02D", "#F57C00"]
    marcador_res = "o" if MODO_TEMPORAL == "hora" else None
    tamanho_marcador = 4 if MODO_TEMPORAL == "hora" else None

    def plotar_graficos_cenario(variaveis_cenario, nome_cenario):
        f_pr_c = variaveis_cenario["f_pr"]
        V_c = variaveis_cenario["V"]
        # f_pr está sempre em m³/min; multiplicar por 60 converte para m³/h em ambos os modos.
        vazao_total = [
            60.0 * sum(f_pr_c[p, r, t].x for p, r in arcos_PR)
            for t in passos
        ]
        vazao_degrau = vazao_total + [vazao_total[-1]]

        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10), sharex=True)

        ax1.fill_between(
            tempo_horas_com_fim, 0, vazao_degrau, step="post",
            color="#2196F3", alpha=0.75, label="Vazão bombeada (m³/h)",
        )
        ax1.axhline(
            Q_ETA_max, color="#0b5394", linestyle="--", linewidth=1.2, alpha=0.8,
            label=f"Capacidade da ETA ({Q_ETA_max} m³/h)",
        )
        ax1.set_ylabel("Vazão total (m³/h)", color="#0b5394", fontweight="bold")
        ax1.tick_params(axis="y", labelcolor="#0b5394")
        ax1.set_ylim(0, Q_ETA_max * 1.12)
        ax1.set_title(
            f"Perfil de bombeamento e tarifa — {nome_cenario} ({DESCRICAO_MODO})",
            fontsize=13.5,
        )
        ax1.grid(True, linestyle="--", alpha=0.4)

        ax1_twin = ax1.twinx()
        ax1_twin.step(
            tempo_horas_com_fim, preco_energia, where="post",
            color="#d32f2f", linewidth=2, label="Tarifa de energia (R$/kWh)",
        )
        ax1_twin.set_ylabel("Tarifa (R$/kWh)", color="#990000", fontweight="bold")
        ax1_twin.tick_params(axis="y", labelcolor="#990000")
        ax1.axvspan(18, 21, color="red", alpha=0.1, label="Horário de ponta (18h–21h)")

        h1, l1 = ax1.get_legend_handles_labels()
        h2, l2 = ax1_twin.get_legend_handles_labels()
        ax1.legend(h1 + h2, l1 + l2, loc="upper left", bbox_to_anchor=(1.08, 1.0))

        for r in reservatorios:
            volume = [param_res[r][2]] + [V_c[r, t].x for t in passos]
            ax2.plot(
                tempo_horas_com_fim, volume, linewidth=1.6,
                marker=marcador_res, markersize=tamanho_marcador,
                color=cores_res[r - 1], label=f"Reservatório {r}",
            )
        ax2.axvspan(18, 21, color="red", alpha=0.1)
        ax2.set_xlabel("Hora do dia (0 = início, 24 = final)", fontweight="bold")
        ax2.set_ylabel("Volume armazenado (m³)", fontweight="bold")
        ax2.set_title(
            f"Dinâmica de carga e descarga dos reservatórios ({DESCRICAO_MODO})",
            fontsize=13.5,
        )
        ax2.set_xticks(range(0, 25, 2))
        ax2.set_xlim(0, 24)
        ax2.legend(loc="upper left", bbox_to_anchor=(1.08, 1.0))
        ax2.grid(True, linestyle="--", alpha=0.5)
        fig.tight_layout()
        plt.show()

        fig_pct, ax_pct = plt.subplots(figsize=(12, 6))
        for r in reservatorios:
            V_max = param_res[r][1]
            volume = [param_res[r][2]] + [V_c[r, t].x for t in passos]
            nivel_percentual = [100.0 * v / V_max for v in volume]
            ax_pct.plot(
                tempo_horas_com_fim, nivel_percentual, linewidth=1.6,
                marker=marcador_res, markersize=tamanho_marcador,
                color=cores_res[r - 1], label=f"Reservatório {r}",
            )
        ax_pct.axvspan(
            18, 21, color="red", alpha=0.1,
            label="Horário de ponta (18h00–20h59)",
        )
        ax_pct.axhline(100, color="black", linestyle=":", alpha=0.5, label="Capacidade máxima (100%)")
        ax_pct.axhline(15, color="#b71c1c", linestyle="--", alpha=0.5, label="Reserva mínima (15%)")
        ax_pct.set_title(
            f"Nível dos reservatórios (%) — {nome_cenario} ({DESCRICAO_MODO})",
            fontsize=13.5,
        )
        ax_pct.set_xlabel("Hora do dia (0 = início, 24 = final)", fontweight="bold")
        ax_pct.set_ylabel("Volume armazenado (%)", fontweight="bold")
        ax_pct.set_xticks(range(0, 25, 2))
        ax_pct.set_xlim(0, 24)
        ax_pct.set_ylim(0, 105)
        ax_pct.grid(True, linestyle="--", alpha=0.7)
        ax_pct.legend(bbox_to_anchor=(1.02, 1.0), loc="upper left")
        fig_pct.tight_layout()
        plt.show()

    def plotar_progresso_otimizacao(df_prog):
        """Plota gráficos separados da evolução de incumbente/limite inferior e do gap."""
        if df_prog is None or df_prog.empty:
            print("Nenhum dado de progresso disponível para plotar.")
            return

        df_plot = df_prog.copy()
        tempo_max_min = float(df_plot["Tempo (min)"].max())
        usar_segundos = tempo_max_min < 2.0
        eixo_x = df_plot["Tempo (min)"] * (60.0 if usar_segundos else 1.0)
        rotulo_x = "Tempo de execução do solver (s)" if usar_segundos else "Tempo de execução do solver (min)"

        mask_primal = df_plot["Melhor custo viável (R$)"].notna()
        mask_dual = df_plot["Limite inferior (R$)"].notna() & (df_plot["Limite inferior (R$)"] > 0)
        mask_gap = df_plot["Gap (%)"].notna() & (df_plot["Limite inferior (R$)"] > 0)

        fig_lim, ax_lim = plt.subplots(figsize=(12, 4.8))
        if mask_primal.any():
            ax_lim.plot(
                eixo_x[mask_primal], df_plot.loc[mask_primal, "Melhor custo viável (R$)"],
                marker="o", linewidth=2.0, color="#006699", label="Melhor custo viável — Incumbente (R$)",
            )
        if mask_dual.any():
            ax_lim.plot(
                eixo_x[mask_dual], df_plot.loc[mask_dual, "Limite inferior (R$)"],
                marker="s", linewidth=2.0, color="#15803D", label="Limite inferior — Dual Bound (R$)",
            )
        ax_lim.set_title(
            f"Progressão da otimização ({SOLVER.upper()} via python-mip · {DESCRICAO_MODO}) — Incumbente vs. Limite Inferior",
            fontsize=13,
        )
        ax_lim.set_xlabel(rotulo_x, fontweight="bold")
        ax_lim.set_ylabel("Custo diário (R$)", fontweight="bold")
        ax_lim.grid(True, linestyle="--", alpha=0.5)
        ax_lim.legend(loc="best")
        fig_lim.tight_layout()
        plt.show()

        fig_gap, ax_gap = plt.subplots(figsize=(12, 4.5))
        if mask_gap.any():
            ax_gap.plot(
                eixo_x[mask_gap], df_plot.loc[mask_gap, "Gap (%)"],
                marker="o", linewidth=2.0, color="#D96B00", label="Gap relativo (%)",
            )
        gap_alvo_pct = obter_gap_alvo_pct()
        if gap_alvo_pct is not None:
            ax_gap.axhline(
                gap_alvo_pct, color="#15803D", linestyle="--", linewidth=1.4,
                label=f"Meta de gap-alvo ({gap_alvo_pct:.2f}%)",
            )
        ax_gap.set_title(
            f"Progressão da otimização ({SOLVER.upper()} via python-mip · {DESCRICAO_MODO}) — Evolução do Gap (%)",
            fontsize=13,
        )
        ax_gap.set_xlabel(rotulo_x, fontweight="bold")
        ax_gap.set_ylabel("Gap relativo (%)", fontweight="bold")
        ax_gap.set_ylim(bottom=0)
        ax_gap.grid(True, linestyle="--", alpha=0.5)
        ax_gap.legend(loc="best")
        fig_gap.tight_layout()
        plt.show()

    plotar_graficos_cenario(vars_model, "Otimização tarifária")
    if base_esta_disponivel():
        plotar_graficos_cenario(vars_base, "Cenário-base (tarifa fixa)")
    if ACOMPANHAR_PROGRESSO:
        plotar_progresso_otimizacao(globals().get("tabela_progresso"))
else:
    print("Execute a otimização com sucesso antes de gerar os gráficos.")